# Results

Plots and tables from saved results only: nothing is trained here. Each results pickle holds one victim
(Hyak: `results/hyak/<RUN_NAME>/<name>_..._fair.pkl`; the other notebooks: `results/victims/`, `results/real/`).

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")                   # the pickles hold src's config classes

import pickle
import warnings
from dataclasses import replace

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
%matplotlib inline

import experiment as ex                     # victim_scores: saved counts and error sums -> accuracy, mse, ...

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 11, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "0.85",
})

In [ ]:
RESULT_FILES = sorted(Path("results/hyak").glob("*/*.pkl"))      # merged pickles, one per victim
# RESULT_FILES = [Path("results/victims/linear_m10_T10_gamma2.95_lam1_fair.pkl"), *Path("results/real").glob("*.pkl")]

METRIC = {"logistic": "accuracy", "regression": "mse", "mlp": "mse"}   # per victim task; a list shows several
# logistic: accuracy | precision | recall | f1        regression, mlp: mse | mae | rmse | r2
DIST_TO = "star"              # star: ||theta - theta*|| (datasets: least-squares fit; mlp has no theta*) | target
FIG_DIR = Path("figures")     # paper figures (pdf); None: don't save
PAPER_SYNTH = ["linear", "logistic"]                 # victims in the synthetic paper figures, when loaded

OURS = "FEVI"
LEARNERS = [OURS, "BDQ", "TRPO"]
EXACT = ex.FEVI_EXACT         # FEVI's value net tested with exact action selection (runs with FEVI_EXACT_TEST=true)
ORDER = ["null", "greedy", "prior", "TRPO", "BDQ", OURS, EXACT]     # prior: untrained TRPO/BDQ (ablation runs only)
COLORS = {"null": "#5b5b5b", "greedy": "#eb6834", "TRPO": "#4a3aa7",
          "BDQ": "#e34948", OURS: "#2a78d6", "prior": "#c9a227", EXACT: "#12a594"}
MARKERS = {"null": "D", "greedy": "s", "TRPO": "^", "BDQ": "X", OURS: "o", "prior": "v", EXACT: "P"}
TITLES = {"wine": "Wine", "airfoil": "Airfoil", "breast_cancer": "Breast cancer", "mnist1": "MNIST (1 vs. rest)"}

In [ ]:
SYNTHETIC = {"regression": "linear", "logistic": "logistic", "mlp": "mlp"}
loaded = []
for path in map(Path, RESULT_FILES):
    with open(path, "rb") as f:
        r = pickle.load(f)
    loaded.append((r["ecfg"].dataset or SYNTHETIC[r["ecfg"].task], r, path))
assert loaded, "no results: set RESULT_FILES"
res, src = {}, {}
for t, r, path in loaded:
    if sum(t == other for other, _, _ in loaded) > 1:     # the same victim in several runs: add the run folder
        t = f"{t} ({path.parent.name})"
    res[t], src[t] = r, path
SHOWN = list(res)
SYNTH = [t for t in SHOWN if not res[t]["ecfg"].dataset]
REAL = [t for t in SHOWN if res[t]["ecfg"].dataset]
NAMES = {t: [n for n in ORDER if n in res[t]["runs"][0]["costs"]] for t in SHOWN}       # every tested policy
LRN = {t: [n for n in LEARNERS if n in res[t]["runs"][0]["logs"]] for t in SHOWN}       # trained learners


def per_seed(t, key):
    return [r[key] for r in res[t]["runs"]]


def panels(n, w=6, h=4.5, ncols=4):
    # one panel per victim, at most ncols per row
    cols = min(n, ncols)
    fig, axes = plt.subplots(-(-n // cols), cols, figsize=(w * cols, h * -(-n // cols)), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat[:n]


def panel_letter(ax, k, y=-0.15):
    ax.text(0.5, y, f"({chr(97 + k)})", transform=ax.transAxes, ha="center", va="top", fontsize=12)


def print_raw(title, rows, t, digits=3):
    n = len(next(iter(rows.values())))
    head = f"{title:26}{'mean':>10}{'std':>9}"
    if n > 1:
        head += "".join(f"{f'seed {s}':>10}" for s in res[t]["seeds"][:n])
    print("\n" + head)
    print("-" * len(head))
    for name, v in rows.items():
        v = np.asarray(v, dtype=float)
        line = f"{name:26}{v.mean():10.{digits}f}{v.std():9.{digits}f}"
        if n > 1:
            line += "".join(f"{x:10.{digits}f}" for x in v)
        print(line)


def saved(t):
    v = res[t]["runs"][0]
    logs = [v["logs"][n] for n in LRN[t]]
    return {"victim models": "victim" in v,
            "per-step victims": "path" in v.get("victim", {}).get("null", {}),
            "validation victims": bool(logs) and "quality" in logs[0]}


for t in SHOWN:
    r = res[t]
    print(f"{t:14} {src[t]}\n{'':14} {len(r['seeds'])} seeds, training episodes {r['episodes']}, "
          f"{r.get('eval_episodes', '?')} test episodes; saved: "
          + (", ".join(k for k, ok in saved(t).items() if ok) or "costs only"))

## Test cost
### Episode cost

In [ ]:
fig, axes = panels(len(SHOWN))
for k, (ax, t) in enumerate(zip(axes, SHOWN)):
    C = {n: np.array([c[n] for c in per_seed(t, "costs")]) for n in NAMES[t]}
    mu = [C[n].mean() for n in NAMES[t]]
    ax.bar(NAMES[t], mu, yerr=[C[n].std() for n in NAMES[t]], capsize=5, color=[COLORS[n] for n in NAMES[t]])
    for i, v in enumerate(mu):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
    ax.axhline(C["null"].mean(), color=COLORS["null"], ls="--", lw=1)
    ax.set_title(t)
    ax.set_ylabel("mean episode cost")
    panel_letter(ax, k)
plt.tight_layout()
plt.show()

for t in SHOWN:
    print_raw(f"[{t}] test cost", {n: [c[n] for c in per_seed(t, "costs")] for n in NAMES[t]}, t)

### Paired comparison against greedy

In [ ]:
def boot_ci(D, n_boot=2000, seed=0):
    # mean and 95% CI, resampling training seeds and test episodes
    g, (S, E) = np.random.default_rng(seed), D.shape
    m = [D[g.integers(0, S, S)][:, g.integers(0, E, E)].mean() for _ in range(n_boot)]
    return D.mean(), *np.percentile(m, [2.5, 97.5])


def episode_costs(t, n):
    return np.array([p[n]["ep"] for p in per_seed(t, "profile")])        # (seeds, test episodes)


for t in SHOWN:
    if "ep" not in per_seed(t, "profile")[0]["null"]:
        print(f"[{t}] no per-episode test costs saved")
        continue
    G = episode_costs(t, "greedy")                                       # same test episodes for all: paired
    head = f"{'[' + t + '] test cost':22}{'mean':>9}{'95% CI':>18}{'- greedy':>11}{'95% CI':>18}"
    print("\n" + head + "\n" + "-" * len(head))
    for n in NAMES[t]:
        X = episode_costs(t, n)
        m, lo, hi = boot_ci(X)
        dm, dlo, dhi = boot_ci(X - G)
        print(f"{n:22}{m:9.3f}  [{lo:6.3f}, {hi:6.3f}]{dm:+11.3f}  [{dlo:+6.3f}, {dhi:+6.3f}]")

### Budget ledger

In [ ]:
def has_budget(t):
    # experiment_real.ipynb pickles don't save cfg and trials
    return {"cfg", "trials"} <= set(res[t]) and set(LEARNERS) <= set(res[t]["hp"]) and LRN[t] == LEARNERS


def budget(t):
    # per-learner budget; per-iteration optimizer updates are fixed by the training loops
    r, cfg, E = res[t], res[t]["cfg"], res[t]["eval_episodes"]
    K, T = r["episodes"][OURS] // r["episodes_per_update"], r["ecfg"].T
    f, b, p = (replace(cfg, **r["hp"][n]) for n in (OURS, "BDQ", "TRPO"))
    extra = int(cfg.myopic_prior)                 # TRPO/BDQ also score the untrained prior as a checkpoint
    return {
        OURS: dict(val=K * E, sim=K * T * cfg.N_episodes * f.fevi_M, step=T * f.fevi_fit_steps,
                   batch=min(cfg.batch_size, cfg.N_episodes)),
        "BDQ": dict(val=(K + extra) * E, sim=0, step=b.n_grad_steps, batch=b.batch_size),
        "TRPO": dict(val=(K + extra) * E, sim=0, step=1 + p.trpo_vf_iters, batch=p.trpo_minibatch),
    }


COLS = ["train eps", "val eps", "model trans.", "updates", "batch", "train s", "total s", "params",
        "tuning runs"]
for t in SHOWN:
    if not has_budget(t):
        print(f"[{t}] no budget ledger: the pickle lacks the method config or tuning counts")
        continue
    r, B = res[t], budget(t)
    K = r["episodes"][OURS] // r["episodes_per_update"]
    head = f"{'[' + t + ']':10}" + "".join(f"{c:>13}" for c in COLS)
    print("\n" + head + "\n" + "-" * len(head))
    for n in LEARNERS:
        row = [r["episodes"][n], B[n]["val"], B[n]["sim"], K * B[n]["step"], B[n]["batch"],
               np.mean([lg[n]["time"][-1] for lg in per_seed(t, "logs")]),
               np.mean([tm[n] for tm in per_seed(t, "times")]),
               r["runs"][0]["params"][n], r["trials"][n]]
        print(f"{n:10}" + "".join(f"{v:13,.0f}" for v in row))
print("\nupdates: optimizer updates over all networks (TRPO: natural-gradient + critic steps); "
      "train s excludes validation; model trans.: transitions FEVI simulates through the known victim")

### Cost over an evaluation episode

In [ ]:
STEP_PANELS = [("cum", r"cumulative cost $J(t) = \sum_{\tau \leq t} g_\tau$"),
               ("pert", r"perturbation $\sum_i |a_{i,t} - 1|$"),
               ("gap", r"$\|\theta_{t+1} - \tilde{\theta}\|$")]

fig, axes = plt.subplots(len(SHOWN), 3, figsize=(18, 4.3 * len(SHOWN)), squeeze=False)
for row, t in enumerate(SHOWN):
    steps = np.arange(res[t]["ecfg"].T)
    every = max(1, len(steps) // 10)
    for col, (key, ylabel) in enumerate(STEP_PANELS):
        ax = axes[row, col]
        for n in NAMES[t]:
            P = np.array([p[n][key] for p in per_seed(t, "profile")])
            mu, sd = P.mean(0), P.std(0)
            ax.plot(steps, mu, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n, markevery=every)
            ax.fill_between(steps, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
        ax.set_ylabel(ylabel)
        ax.set_xticks(steps[::every])
        if row == len(SHOWN) - 1:
            ax.set_xlabel("attack step t")
    axes[row, 0].set_title(t, loc="left", fontweight="bold")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

for t in SHOWN:
    steps = np.arange(res[t]["ecfg"].T)
    cols = np.unique(np.r_[steps[::max(1, len(steps) // 10)], steps[-1]])   # at most ~11 steps
    print(f"\n[{t}] J(t){'':5}" + "".join(f"{f't={s}':>8}" for s in cols))
    for n in NAMES[t]:
        J = np.mean([p[n]["cum"] for p in per_seed(t, "profile")], axis=0)
        print(f"{n:14}" + "".join(f"{v:8.2f}" for v in J[cols]))

In [ ]:
MODE = "vs_null"   # "raw" | "vs_null": J(t) / J_null(t) | "vs_greedy": J(t) - J_greedy(t)
YLABEL = {"raw": r"cumulative cost $J(t) = \sum_{\tau \leq t} g_\tau$",
          "vs_null": r"$J(t)\,/\,J_{\mathrm{null}}(t)$",
          "vs_greedy": r"$J(t) - J_{\mathrm{greedy}}(t)$"}
STYLES = {"null": "--", "greedy": "-.", "prior": ":", "TRPO": "-", "BDQ": "-", OURS: "-", EXACT: "--"}


def relative(J, n):
    if MODE == "vs_null":
        return J[n] / J["null"]
    if MODE == "vs_greedy":
        return J[n] - J["greedy"]
    return J[n]


fig, axes = panels(len(SHOWN), w=6.5)
for k, (ax, t) in enumerate(zip(axes, SHOWN)):
    steps = np.arange(res[t]["ecfg"].T)
    every = max(1, len(steps) // 10)
    J = {n: np.array([p[n]["cum"] for p in per_seed(t, "profile")]) for n in NAMES[t]}   # (seeds, T)
    for n in NAMES[t]:
        Y = relative(J, n)
        mu, sd = Y.mean(0), Y.std(0)
        ax.plot(steps, mu, STYLES[n], marker=MARKERS[n], color=COLORS[n], lw=2, ms=6, label=n, markevery=every)
        ax.fill_between(steps, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
    if MODE == "vs_greedy":
        ax.axhline(0, color="k", lw=0.8)
    ax.set_title(t)
    ax.set_xlabel("attack step t")
    ax.set_xticks(steps[::every])
    ax.set_ylabel(YLABEL[MODE])
    panel_letter(ax, k, -0.18)
axes[0].legend()
plt.tight_layout()
plt.show()

for t in SHOWN:
    steps = np.arange(res[t]["ecfg"].T)
    cols = np.unique(np.r_[steps[::max(1, len(steps) // 10)], steps[-1]])
    J = {n: np.array([p[n]["cum"] for p in per_seed(t, "profile")]) for n in NAMES[t]}
    print(f"\n[{t}] {MODE}{'':3}" + "".join(f"{f't={s}':>8}" for s in cols))
    for n in NAMES[t]:
        print(f"{n:14}" + "".join(f"{v:8.3f}" for v in relative(J, n).mean(0)[cols]))

### Across edit-cost weights

For victims loaded at several $\lambda$ (`scripts/hyak/lam_sweep.sh`). Raw cost rises with $\lambda$ for every
policy, because $\lambda$ prices the edits, so each panel compares the policies at the same $\lambda$: cost relative
to null, the distance part of the cost, and the edits made. Bands: ± 1 std over training seeds.

In [ ]:
sys.path.insert(0, "scripts")
import lam_summary                          # the table and figure lam_sweep.sh's summary job writes

swept = {name: rs for name, rs in lam_summary.load(RESULT_FILES).items() if len({r["lam"] for r in rs}) > 1}
if not swept:
    print("no victim loaded at more than one lambda")
for name, rs in swept.items():
    rows = lam_summary.rows(name, rs)
    lam_summary.plot(rows, FIG_DIR / f"cost_vs_lambda_{name}" if FIG_DIR else None)
    plt.show()
    lam_summary.table(rows)

## Training
### Learning curves

In [ ]:
PANELS = [("cost", "validation episode cost"),
          ("pert", r"perturbation $\sum_i |a_i - 1|$ per step"),
          ("gap", r"$\|\theta_t - \tilde{\theta}\|$")]


def curve_stats(t, name, key):
    H = np.array([lg[name][key] for lg in per_seed(t, "logs")])
    return H.mean(0), H.std(0)


def train_x(t, n_points):
    # training episodes at each validation checkpoint (one per update)
    return np.arange(1, n_points + 1) * res[t]["episodes_per_update"]


fig, axes = plt.subplots(len(SHOWN), 3, figsize=(18, 4.3 * len(SHOWN)), squeeze=False)
for row, t in enumerate(SHOWN):
    ref = {n: np.mean([r[n] for r in per_seed(t, "ref")], axis=0) for n in ["null", "greedy"]}
    for col, (key, ylabel) in enumerate(PANELS):
        ax = axes[row, col]
        for n in LRN[t]:
            mu, sd = curve_stats(t, n, key)
            x = train_x(t, len(mu))
            ax.plot(x, mu, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n, markevery=max(1, len(x) // 8))
            ax.fill_between(x, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
        ax.axhline(ref["null"][col], color=COLORS["null"], ls="--", lw=1.5, label="null")
        ax.axhline(ref["greedy"][col], color=COLORS["greedy"], ls="-.", lw=1.5, label="greedy")
        ax.set_ylabel(ylabel)
        if row == len(SHOWN) - 1:
            ax.set_xlabel("training episodes")
    axes[row, 0].set_title(t, loc="left", fontweight="bold")
axes[0, 0].legend(loc="upper right")
plt.tight_layout()
plt.show()

for t in SHOWN:
    print_raw(f"[{t}] val. cost, last iter", {n: [h[n][-1] for h in per_seed(t, "hist")] for n in LRN[t]}, t)

### Wall-clock time

In [ ]:
fig, axes = panels(len(SHOWN))
for k, (ax, t) in enumerate(zip(axes, SHOWN)):
    ref = {n: np.mean([r[n] for r in per_seed(t, "ref")], axis=0) for n in ["null", "greedy"]}
    for n in LRN[t]:
        mu_t, _ = curve_stats(t, n, "time")
        mu, sd = curve_stats(t, n, "cost")
        ax.plot(mu_t, mu, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n, markevery=max(1, len(mu) // 8))
        ax.fill_between(mu_t, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
    ax.axhline(ref["null"][0], color=COLORS["null"], ls="--", lw=1.5, label="null")
    ax.axhline(ref["greedy"][0], color=COLORS["greedy"], ls="-.", lw=1.5, label="greedy")
    ax.set_xscale("log")
    ax.set_xlabel("wall-clock training time (s)")
    ax.set_ylabel("validation episode cost")
    ax.set_title(t)
    panel_letter(ax, k, -0.18)
axes[0].legend()
plt.tight_layout()
plt.show()

for t in SHOWN:
    print_raw(f"[{t}] training time (s)", {n: [tm[n] for tm in per_seed(t, "times")] for n in LRN[t]}, t, 1)

### Optimizer updates

In [ ]:
BUDGETED = [t for t in SHOWN if has_budget(t)]
if BUDGETED:
    fig, axes = panels(len(BUDGETED))
    for k, (ax, t) in enumerate(zip(axes, BUDGETED)):
        ref = {n: np.mean([r[n] for r in per_seed(t, "ref")], axis=0) for n in ["null", "greedy"]}
        B = budget(t)
        for n in LEARNERS:
            mu, sd = curve_stats(t, n, "cost")
            x = np.arange(1, len(mu) + 1) * B[n]["step"]
            ax.plot(x, mu, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n, markevery=max(1, len(mu) // 8))
            ax.fill_between(x, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
        ax.axhline(ref["null"][0], color=COLORS["null"], ls="--", lw=1.5, label="null")
        ax.axhline(ref["greedy"][0], color=COLORS["greedy"], ls="-.", lw=1.5, label="greedy")
        ax.set_xscale("log")
        ax.set_xlabel("optimizer updates (all networks)")
        ax.set_ylabel("validation episode cost")
        ax.set_title(t)
        panel_letter(ax, k, -0.18)
    axes[0].legend()
    plt.tight_layout()
    plt.show()
else:
    print("no victim has a budget ledger")

## Paper figures

In [ ]:
PAPER_WIDTH = 5.5                      # ICLR \textwidth in inches: draw at print size, no rescaling


def paper_rc(font):
    # 6.5: synthetic figures; 7.5: real-data figure
    return {"font.size": font, "axes.labelsize": font, "axes.titlesize": font + 0.5, "xtick.labelsize": font - 1,
            "ytick.labelsize": font - 1, "legend.fontsize": font - 0.5, "axes.linewidth": 0.5,
            "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2,
            "ytick.major.size": 2, "xtick.major.pad": 1.5, "ytick.major.pad": 1.5,
            "axes.labelpad": 2, "grid.linewidth": 0.3, "pdf.fonttype": 42}


def paper_lines(scale):
    # line widths per policy (ours thicker); scale 1: synthetic figures, 1.2: real-data figure
    w = {"null": 0.8, "greedy": 0.9, "TRPO": 0.9, "BDQ": 0.9, OURS: 1.6} if scale == 1 else \
        {"null": 1.0, "greedy": 1.1, "TRPO": 1.1, "BDQ": 1.1, OURS: 1.9}
    ls = {"null": "--", "greedy": "-."}
    return {n: dict(ls=ls.get(n, "-"), lw=v) for n, v in w.items()}


LINE, MS = paper_lines(1), (3.0, 2.2)          # set again by each figure below


def style(n, **kw):
    # ours: thicker and on top; the others slightly faded
    ours = n == OURS
    return dict(color=COLORS[n], marker=MARKERS[n], ms=MS[0] if ours else MS[1], mec="white", mew=0.3,
                zorder=4 if ours else (3 if n in LEARNERS else 2), alpha=1.0 if ours else 0.75,
                **LINE[n], **kw)


def line(ax, x, y, n, **kw):
    # ours gets a thin white underlay, so it stays visible where lines cross
    if n == OURS:
        ax.plot(x, y, color="white", lw=LINE[n]["lw"] + 1.4, zorder=3.9, solid_capstyle="round")
    return ax.plot(x, y, **style(n, **kw))


def band(n):
    return dict(color=COLORS[n], alpha=0.22 if n == OURS else 0.1, lw=0)


def plot_cumulative(ax, t):
    # cumulative test cost J(t) per attack step
    steps = np.arange(res[t]["ecfg"].T)
    for n in [n for n in NAMES[t] if n in LINE]:
        P = np.array([p[n]["cum"] for p in per_seed(t, "profile")])
        mu, sd = P.mean(0), P.std(0)
        line(ax, steps, mu, n, label=n)
        ax.fill_between(steps, mu - sd, mu + sd, **band(n))
    ax.set_xlabel("attack step $t$")
    ax.set_xticks(steps[::3])
    ax.yaxis.set_major_locator(MaxNLocator(4))


def plot_validation(ax, t):
    # validation episode cost after each update, with null / greedy as reference lines
    ref = {n: np.mean([r[n] for r in per_seed(t, "ref")], axis=0)[0] for n in ["null", "greedy"]}
    for n in ["null", "greedy"]:
        ax.axhline(ref[n], color=COLORS[n], zorder=2, **LINE[n])
    for n in LRN[t]:
        H = np.array([lg[n]["cost"] for lg in per_seed(t, "logs")])
        mu, sd = H.mean(0), H.std(0)
        x = train_x(t, len(mu))
        line(ax, x, mu, n, markevery=max(1, len(x) // 5))
        ax.fill_between(x, mu - sd, mu + sd, **band(n))
    ax.set_xlabel("training episodes")
    ax.xaxis.set_major_locator(MaxNLocator(3, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.set_ylabel("validation cost")


def finish_legend(ax, handles, labels, loc, size):
    labels = [f"{l} (ours)" if l == OURS else l for l in labels]
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.35 * (hi - lo))                     # headroom for the legend
    leg = ax.legend(handles, labels, loc=loc, handlelength=1.2, handletextpad=0.3, borderpad=0.3,
                    labelspacing=0.25, borderaxespad=0.3, frameon=True, fancybox=False, edgecolor="0.6",
                    framealpha=1.0, prop={"size": size})
    leg.get_frame().set_linewidth(0.5)
    for text in leg.get_texts():
        if text.get_text().endswith("(ours)"):
            text.set_fontweight("bold")


def save(fig, name):
    if FIG_DIR:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / name, bbox_inches="tight")

In [ ]:
PAPER = [t for t in PAPER_SYNTH if t in res]
LINE, MS = paper_lines(1), (3.0, 2.2)
if PAPER:
    with plt.rc_context(paper_rc(6.5)):
        fig, axes = plt.subplots(1, 2 * len(PAPER), figsize=(PAPER_WIDTH, 1.75))
        for row, t in enumerate(PAPER):
            plot_cumulative(axes[2 * row], t)
            axes[2 * row].set_ylabel(r"Cumulative cost $\widetilde{J}(t)$")
            plot_validation(axes[2 * row + 1], t)
            axes[2 * row].set_title(t, loc="left", fontweight="bold", pad=3)
        for k, ax in enumerate(axes):
            ax.text(0.5, -0.36, f"({chr(97 + k)})", transform=ax.transAxes, ha="center", va="top")
        finish_legend(axes[-1], *axes[0].get_legend_handles_labels(), "upper right", 5)
        fig.tight_layout(pad=0.2, w_pad=0.6)
        save(fig, "cumulative_and_validation.pdf")
        plt.show()
else:
    print(f"none of PAPER_SYNTH {PAPER_SYNTH} loaded")

In [ ]:
LINE, MS = paper_lines(1.2), (3.6, 2.8)
if REAL:
    with plt.rc_context(paper_rc(7.5)):
        fig, axes = plt.subplots(1, len(REAL), figsize=(PAPER_WIDTH, 1.75), squeeze=False)
        axes = axes[0]
        for k, (ax, t) in enumerate(zip(axes, REAL)):
            plot_cumulative(ax, t)
            ax.set_title(TITLES.get(t, t), loc="left", fontweight="bold", pad=3)
            ax.text(0.5, -0.31, f"({chr(97 + k)})", transform=ax.transAxes, ha="center", va="top")
        axes[0].set_ylabel(r"Cumulative cost $\widetilde{J}(t)$")
        finish_legend(axes[0], *axes[0].get_legend_handles_labels(), "upper left", 5.5)
        fig.tight_layout(pad=0.2, w_pad=0.8)
        save(fig, "cumulative_real.pdf")
        plt.show()
else:
    print("no dataset victims loaded")

In [ ]:
WC_LW = {"TRPO": 0.9, "BDQ": 0.9, OURS: 1.6}
if PAPER:
    with plt.rc_context(paper_rc(6.5)):
        fig, axes = plt.subplots(1, len(PAPER), figsize=(PAPER_WIDTH, 1.9), squeeze=False)
        for k, (ax, t) in enumerate(zip(axes[0], PAPER)):
            for n in LRN[t]:
                H = np.array([lg[n]["time"] for lg in per_seed(t, "logs")])     # cumulative training seconds
                mu, sd = H.mean(0), H.std(0)
                x = train_x(t, len(mu))
                ours = n == OURS
                ax.plot(x, mu, "-", marker=MARKERS[n], color=COLORS[n], lw=WC_LW[n], ms=3.0 if ours else 2.2,
                        mec="white", mew=0.3, markevery=max(1, len(x) // 5), alpha=1.0 if ours else 0.75,
                        zorder=4 if ours else 3, label=f"{n} (ours)" if ours else n)
                ax.fill_between(x, mu - sd, mu + sd, color=COLORS[n], alpha=0.22 if ours else 0.1, lw=0)
            ax.set_yscale("log")
            ax.xaxis.set_major_locator(MaxNLocator(3, integer=True))
            ax.set_xlabel("training episodes")
            ax.set_ylabel("wall-clock training time (s)")
            ax.set_title(f"({chr(97 + k)}) {t}", loc="left", pad=3)
            ax.grid(True, axis="y", which="major")
        leg = axes[0, 0].legend(loc="lower right")
        for text in leg.get_texts():
            if text.get_text().endswith("(ours)"):
                text.set_fontweight("bold")
        fig.tight_layout(pad=0.2, w_pad=1.0)
        save(fig, "wallclock_vs_episodes.pdf")
        plt.show()

for t in SHOWN:
    print(f"[{t}] training seconds at the end (mean ± std over seeds):  "
          + "  ".join(f"{n}={np.mean([lg[n]['time'][-1] for lg in per_seed(t, 'logs')]):.0f}"
                      f"±{np.std([lg[n]['time'][-1] for lg in per_seed(t, 'logs')]):.0f}" for n in LRN[t]))

## Victim quality on clean data

The victim models each policy leaves behind, scored on clean data the attack never touches (synthetic victims:
10,000 fresh draws; datasets: the held-out rows; mnist: its test images). `METRIC` picks the score and
`DIST_TO` the parameter the distance is measured to: $\theta^*$, the true parameter (datasets: the least-squares
fit on the pool), or the attack target $\tilde\theta$. Each point is a mean over episodes; bands: ± 1 std over
training seeds. Black lines: $\theta^*$ and the target, drawn when they fall near the curves (the tables list
every value).

In [ ]:
METRIC_LABEL = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "F1",
                "mse": "MSE", "mae": "MAE", "rmse": "RMSE", "r2": "$R^2$"}
REF_TEX = {"star": r"\theta^*", "target": r"\tilde{\theta}"}
REF_LINE = {"star": dict(color="k", ls=":", lw=1.3, label=r"$\theta^*$"),
            "target": dict(color="k", ls=(0, (6, 2, 1, 2, 1, 2)), lw=1.0, label=r"target $\tilde{\theta}$")}
assert DIST_TO in REF_TEX, f"DIST_TO {DIST_TO!r}: star or target"


def metrics(t):
    task = res[t]["ecfg"].task
    allowed = ("accuracy", "precision", "recall", "f1") if task == "logistic" else ("mse", "mae", "rmse", "r2")
    m = [METRIC[task]] if isinstance(METRIC[task], str) else list(METRIC[task])
    assert set(m) <= set(allowed), f"METRIC[{task!r}] = {METRIC[task]}: choose from {allowed}"
    return m


def nanmean(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)       # e.g. precision with no positive predictions
        return np.nanmean(np.asarray(a, dtype=float), axis=axis)


def nanstd(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanstd(np.asarray(a, dtype=float), axis=axis)


def mean_sd(v):
    # (seeds, columns) -> "mean ± std" per column
    return "".join(f"{'n/a' if np.isnan(m) else f'{m:.4f} ± {s:.4f}':>20}" for m, s in zip(nanmean(v, 0), nanstd(v, 0)))


def score(s, m):
    # per-model metric m from saved confusion counts or error sums
    return np.asarray(ex.victim_scores(s)[m], dtype=float)


def dist(s):
    # per-model saved distance to the DIST_TO parameter
    return np.asarray(s[f"{DIST_TO}_dist"], dtype=float)


def ref_theta(v):
    # the DIST_TO parameter, or None when not saved (mlp has no theta*)
    return v[DIST_TO]["theta"][0] if DIST_TO in v else None


def dist_to_ref(v, theta):
    r = ref_theta(v)
    return np.full(len(theta), np.nan) if r is None else np.linalg.norm(theta - r, axis=-1)


def columns(t, sub):
    return [("dist", rf"$\|\theta_{sub} - {REF_TEX[DIST_TO]}\|$")] + \
        [(m, f"test {METRIC_LABEL[m]}") for m in metrics(t)]


def table_head(title, t):
    names = [f"|θ_T - {'θ*' if DIST_TO == 'star' else 'θ~'}|"] + [METRIC_LABEL[m].strip("$") for m in metrics(t)]
    head = f"{title:24}" + "".join(f"{c:>20}" for c in names)
    print("\n" + head + "\n" + "-" * len(head))


def reference_lines(ax, t, key):
    # theta* and the target (their scores; their distance apart), when near the curves: far ones flatten them
    v = res[t]["runs"][0]["victim"]
    lo, hi = ax.get_ylim()
    for name in ("star", "target"):
        if name not in v:
            continue
        if key != "dist":
            y = score(v[name], key)[0]
        elif name != DIST_TO and DIST_TO in v:
            y = dist_to_ref(v, v[name]["theta"])[0]
        else:
            continue
        if lo - 0.25 * (hi - lo) <= y <= hi + 0.25 * (hi - lo):
            ax.axhline(y, zorder=1, **REF_LINE[name])


def figure_rows(rows, ncol):
    fig, axes = plt.subplots(len(rows), ncol, figsize=(6 * ncol, 4.3 * len(rows)), squeeze=False)
    return fig, axes


def finish_rows(axes, rows, xlabel):
    for row, t in enumerate(rows):
        n_used = len(columns(t, ""))
        for ax in axes[row, n_used:]:
            ax.set_visible(False)
        for ax in axes[row, :n_used]:
            if not any(np.isfinite(np.asarray(l.get_ydata(), dtype=float)).any() for l in ax.lines):
                ax.cla()
                ax.axis("off")
                ax.text(0.5, 0.5, f"no ${REF_TEX[DIST_TO]}$ for this victim", transform=ax.transAxes,
                        ha="center", va="center", color="0.4")
            elif row == len(rows) - 1 or not axes[row + 1, list(axes[row]).index(ax)].get_visible():
                ax.set_xlabel(xlabel)
        axes[row, 0].set_title(t, loc="left", fontweight="bold")
    seen = {}                                     # one legend for every panel's lines
    for ax in axes.flat:
        for h, l in zip(*ax.get_legend_handles_labels()):
            seen.setdefault(l, h)
    fig = axes[0, 0].figure
    fig.legend(seen.values(), seen.keys(), loc="upper center", ncol=len(seen), bbox_to_anchor=(0.5, 1))
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.45 / fig.get_figheight()))
    plt.show()


VICTIMS_SAVED = [t for t in SHOWN if saved(t)["victim models"]]
for t in SHOWN:
    if t not in VICTIMS_SAVED:
        print(f"[{t}] no victim models saved (results from before they were added)")
for t in VICTIMS_SAVED:
    runs, ms = per_seed(t, "victim"), metrics(t)
    rows = {}
    for n, label in (("initial", "initial θ0"), ("star", "θ*"), ("target", "target θ~")):
        if n in runs[0]:
            rows[label] = [[dist_to_ref(v, v[n]["theta"])[0]] + [score(v[n], m)[0] for m in ms] for v in runs]
    for n in NAMES[t]:                                  # each seed: mean over its test episodes
        rows[n] = [[nanmean(dist_to_ref(v, v[n]["theta"]))] + [nanmean(score(v[n], m)) for m in ms] for v in runs]
    table_head(f"[{t}] test victims", t)
    for name, v in rows.items():
        print(f"{name:24}" + mean_sd(v))
print("\nlearners and references: final victims of the test episodes (mean ± std over training seeds)")

### During training

At every validation checkpoint (seed 998 episodes, one checkpoint per update): the final victims' distance to
the reference parameter and their test scores. Dashed: null and greedy on the same episodes.

In [ ]:
VAL = [t for t in SHOWN if saved(t)["validation victims"]]
for t in SHOWN:
    if t not in VAL:
        print(f"[{t}] no validation victims saved: rerun with the current code")


def checkpoint_value(q, key):
    # mean over the checkpoint's validation episodes
    return nanmean(dist(q) if key == "dist" else score(q, key))


if VAL:
    fig, axes = figure_rows(VAL, max(len(columns(t, "T")) for t in VAL))
    for row, t in enumerate(VAL):
        runs = per_seed(t, "victim")
        for col, (key, ylabel) in enumerate(columns(t, "T")):
            ax = axes[row, col]
            for n in LRN[t]:
                H = np.array([[checkpoint_value(q, key) for q in lg[n]["quality"]] for lg in per_seed(t, "logs")])
                mu, sd = nanmean(H, 0), nanstd(H, 0)                       # over seeds
                x = train_x(t, H.shape[1])
                ax.plot(x, mu, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n, markevery=max(1, len(x) // 8))
                ax.fill_between(x, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
            for n, ls in (("null", "--"), ("greedy", "-.")):
                if "val" in runs[0].get(n, {}):
                    ax.axhline(nanmean([checkpoint_value(v[n]["val"], key) for v in runs]), color=COLORS[n],
                               ls=ls, lw=1.5, label=n)
            reference_lines(ax, t, key)
            ax.set_ylabel(ylabel)
    finish_rows(axes, VAL, "training episodes")

for t in VAL:
    cols = columns(t, "T")
    table_head(f"[{t}] kept checkpoint", t)
    runs, logs = per_seed(t, "victim"), per_seed(t, "logs")
    rows = {n: [[checkpoint_value(v[n]["val"], k) for k, _ in cols] for v in runs]
            for n in ("null", "greedy") if "val" in runs[0].get(n, {})}
    for n in LRN[t]:                                  # the checkpoint each seed keeps: lowest validation cost
        rows[n] = [[checkpoint_value(lg[n]["quality"][int(np.argmin(lg[n]["cost"]))], k) for k, _ in cols]
                   for lg in logs]
    for name, v in rows.items():
        print(f"{name:24}" + mean_sd(v))
print("\nvalidation victims at the checkpoint each learner keeps (mean ± std over training seeds)")

### Over an attack episode

The test victims after each update $t$ ($t = 0$: the shared start $\theta_0$; $t = T$: the models in the table
above).

In [ ]:
PATHS = [t for t in SHOWN if saved(t)["per-step victims"]]
for t in SHOWN:
    if t not in PATHS:
        print(f"[{t}] no per-step victims saved: rerun with the current code")

if PATHS:
    fig, axes = figure_rows(PATHS, max(len(columns(t, "t")) for t in PATHS))
    for row, t in enumerate(PATHS):
        runs = per_seed(t, "victim")
        steps = np.arange(res[t]["ecfg"].T + 1)
        every = max(1, len(steps) // 10)
        for col, (key, ylabel) in enumerate(columns(t, "t")):
            ax = axes[row, col]
            for n in NAMES[t]:
                P = []
                for v in runs:                                   # theta_0, then the mean over test episodes per step
                    start = dist_to_ref(v, v["initial"]["theta"])[0] if key == "dist" else score(v["initial"], key)[0]
                    per = dist(v[n]["path"]) if key == "dist" else score(v[n]["path"], key)     # (episodes, T)
                    P.append(np.r_[start, nanmean(per, 0)])
                mu, sd = nanmean(P, 0), nanstd(P, 0)
                ax.plot(steps, mu, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n, markevery=every)
                ax.fill_between(steps, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
            reference_lines(ax, t, key)
            ax.set_xticks(steps[::every])
            ax.set_ylabel(ylabel)
    finish_rows(axes, PATHS, "victim updates $t$")